1. **Title**

Day 12 - Used Car Data Preprocessing

This notebook preprocesses a used car dataset by handling outliers, encoding categorical variables, scaling numerical features, and preparing the data for machine learning while avoiding data leakage.


2. **Import libraries**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

3. **Load the Dataset**

In [2]:
df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")
df.head()

,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


4. **Understand the dataset**

In [3]:
print("Shape:", df.shape)
print(df.dtypes)

Shape: (320, 15)
Car_ID                 object
Brand                  object
Year                    int64
Mileage_Km              int64
Engine_CC               int64
Power_BHP             float64
Fuel_Type              object
Transmission           object
City                   object
Seller_Type            object
Condition              object
Previous_Owners         int64
Accidents_Reported      int64
Service_Score           int64
Resale_Price_Lakh     float64
dtype: object


In [4]:
df.isnull().sum()

,0
Car_ID,0
Brand,0
Year,0
Mileage_Km,0
Engine_CC,0
Power_BHP,0
Fuel_Type,0
Transmission,0
City,0
Seller_Type,0


In [5]:
df.describe()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000
mean,2019.537500,74110.203125,1346.703125,150.489688,1.668750,0.243750,76.203125,4.963031
std,3.341367,38885.260771,543.408160,36.665353,0.865369,0.528164,12.745864,3.359259
min,2014.000000,700.000000,600.000000,51.400000,1.000000,0.000000,55.000000,1.200000
25%,2017.000000,46323.250000,1004.750000,128.450000,1.000000,0.000000,64.750000,2.277500
50%,2020.000000,72718.500000,1303.000000,150.750000,1.000000,0.000000,77.000000,4.610000
75%,2022.000000,97951.500000,1635.250000,171.475000,2.000000,0.000000,87.000000,6.835000
max,2025.000000,320000.000000,5000.000000,390.000000,4.000000,2.000000,98.000000,28.500000


5. **Separate Features and Target**

In [7]:
X = df.drop(["Car_ID", "Resale_Price_Lakh"], axis=1)
y = df["Resale_Price_Lakh"]

6. **Train-Test Split**

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (256, 13)
Testing data: (64, 13)


7. **Outlier Detection using IQR**

In [9]:
numeric_columns = X_train.select_dtypes(include="number").columns

for column in numeric_columns:
    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((X_train[column] < lower) | (X_train[column] > upper)).sum()

    print(column, ":", count, "outliers")

Year : 0 outliers
Mileage_Km : 2 outliers
Engine_CC : 6 outliers
Power_BHP : 5 outliers
Previous_Owners : 10 outliers
Accidents_Reported : 47 outliers
Service_Score : 0 outliers


8. **Handle Outliers**

In [10]:
for column in numeric_columns:
    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X_train[column] = X_train[column].clip(lower, upper)
    X_test[column] = X_test[column].clip(lower, upper)

9. **Encode Ordinal Data**

In [11]:
condition_map = {
    "Poor": 1,
    "Average": 2,
    "Good": 3,
    "Very Good": 4,
    "Excellent": 5
}

X_train["Condition"] = X_train["Condition"].map(condition_map)
X_test["Condition"] = X_test["Condition"].map(condition_map)

10. **Select Categorical and Numerical Features**



In [12]:
categorical_columns = [
    "Brand",
    "Fuel_Type",
    "Transmission",
    "City",
    "Seller_Type"
]

numeric_columns = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]



11. **One-Hot Encoding and Scaling**

In [13]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_columns),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns)
])

12. **Fit Only on Training Data**

In [14]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

13. **Create Processed DataFrames**

In [15]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [16]:
X_train_processed["Resale_Price_Lakh"] = y_train
X_test_processed["Resale_Price_Lakh"] = y_test

14. **Verify Processed Data**

In [17]:
X_train_processed.head()

,num__Year,num__Mileage_Km,num__Engine_CC,num__Power_BHP,num__Previous_Owners,num__Accidents_Reported,num__Service_Score,cat__Brand_Honda,cat__Brand_Hyundai,cat__Brand_Kia,...,cat__City_Hyderabad,cat__City_Jaipur,cat__City_Kochi,cat__City_Lucknow,cat__City_Mumbai,cat__City_Pune,cat__Seller_Type_Certified Dealer,cat__Seller_Type_Dealer,cat__Seller_Type_Individual,Resale_Price_Lakh
132,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.26
317,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.30
234,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.23
312,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7.09
232,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.69


In [18]:
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (256, 37)
Testing shape: (64, 37)


In [19]:
X_train_processed.isnull().sum().sum()

np.int64(0)

15. **Combine for Final Processed Dataset**

In [20]:
processed_data = pd.concat([
    X_train_processed.assign(Dataset="Train"),
    X_test_processed.assign(Dataset="Test")
])

processed_data = processed_data.reset_index(drop=True)

processed_data.head()

,num__Year,num__Mileage_Km,num__Engine_CC,num__Power_BHP,num__Previous_Owners,num__Accidents_Reported,num__Service_Score,cat__Brand_Honda,cat__Brand_Hyundai,cat__Brand_Kia,...,cat__City_Jaipur,cat__City_Kochi,cat__City_Lucknow,cat__City_Mumbai,cat__City_Pune,cat__Seller_Type_Certified Dealer,cat__Seller_Type_Dealer,cat__Seller_Type_Individual,Resale_Price_Lakh,Dataset
0,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.26,Train
1,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.30,Train
2,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.23,Train
3,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7.09,Train
4,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.69,Train


16. **Export the Dataset**

In [21]:
processed_data.to_csv(
    "preprocessed_used_car_dataset.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")

Preprocessed dataset saved successfully!


In [22]:
from google.colab import files
files.download("preprocessed_used_car_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>